# 04 — Embedding Model Comparison

Evaluate all candidate embedding models on downstream clustering quality — not generic NLP benchmarks.
For instruction-tuned models, compare multiple instruction variants as an additional axis.

Each (model × instruction) combination is cached independently.
Re-running this notebook is fast: already-computed embeddings are loaded from disk.

## Model tiers

### CPU tier (16 GB RAM, no CUDA — run locally)
| Model | Params | Instruction? |
|---|---|---|
| `all-MiniLM-L6-v2` | 22 M | no |
| `all-mpnet-base-v2` | 110 M | no |
| `hkunlp/instructor-base` | 110 M | yes |
| `hkunlp/instructor-large` | 335 M | yes |
| `intfloat/multilingual-e5-large-instruct` | 560 M | yes |
| `Qwen/Qwen3-Embedding-0.6B` | 600 M | yes |
| `BAAI/bge-m3` | 570 M | no |

### GPU tier (university server — enable below)
| Model | Params |
|---|---|
| `hkunlp/instructor-xl` | 1.5 B |
| `Qwen/Qwen3-Embedding-4B` | 4 B |
| `intfloat/e5-mistral-7b-instruct` | 7 B |

**Prerequisite:** Run `02_business_data_exploration.ipynb` first.

## Setup

In [ ]:
import sys, json, time
import numpy as np
import matplotlib.pyplot as plt
sys.path.insert(0, '..')

from utils import (
    load_config, get_preprocessor,
    embedding_path, save_array, load_array, array_exists,
    compute_metrics, compute_coherence, compute_rating_entropy,
    log_result,
)

# ── Edit these to switch between CPU small / CPU medium / GPU large ───────────
cfg = load_config(sample_size=5_000, device="cpu", batch_size=64)
# ─────────────────────────────────────────────────────────────────────────────

cfg.ensure_dirs()
print(cfg)

# Set False for fast iteration during the embedding sweep — coherence adds ~30s per model
COMPUTE_COHERENCE = True

In [ ]:
# Load reviews AND star ratings (needed for Tier 3 rating entropy)
preprocess = get_preprocessor("minimal")
texts = []
stars = []
with open(cfg.data_path) as f:
    for line in f:
        obj = json.loads(line)
        texts.append(preprocess(obj["text"]))
        stars.append(float(obj.get("stars", float("nan"))))
        if len(texts) >= cfg.sample_size:
            break

stars = np.array(stars)
print(f"Loaded {len(texts):,} reviews  |  stars missing: {int(np.isnan(stars).sum())}")

## Instruction variants

Defined once here; referenced by every instruction-aware model below.

In [ ]:
INSTRUCTIONS = {
    "no_inst":  None,
    "generic":  "Represent the hotel review for topic clustering:",
    "domain":   "Represent the hotel review for clustering by theme (room quality, staff, location, breakfast, cleanliness, value):",
    "sentiment": "Represent the hotel review to capture the main sentiment and opinion:",
}

## Helper: fixed evaluation pipeline

Used for fair comparison across all models: UMAP(nc=10, nn=15) + HDBSCAN(mcs=15, ms=5).

In [ ]:
import umap
import hdbscan

def evaluate_embeddings(embeddings, texts, stars, seed=42, compute_coh=True):
    """
    Fixed UMAP(10d) + HDBSCAN → three-tier metrics dict.

    Using a fixed config here ensures that differences in the returned metrics
    are attributable to the embedding model, not to DR/clustering hyperparameters.
    The chosen fixed config (nc=10, nn=15, mcs=15, ms=5) was selected in nb 05/06.
    """
    t0 = time.time()
    reducer = umap.UMAP(n_components=10, n_neighbors=15, min_dist=0.0,
                         metric="cosine", random_state=seed)
    reduced = reducer.fit_transform(embeddings)
    labels = hdbscan.HDBSCAN(min_cluster_size=15, min_samples=5).fit_predict(reduced)
    elapsed = time.time() - t0

    metrics = compute_metrics(reduced, labels, runtime_s=elapsed, seed=seed)
    coherence = compute_coherence(texts, labels) if compute_coh else None
    entropy   = compute_rating_entropy(stars, labels)
    return reduced, labels, elapsed, coherence, entropy

FIXED_DR_CONFIG = dict(reduction_method="umap", umap_n_components=10,
                        umap_n_neighbors=15, umap_min_dist=0.0, umap_metric="cosine")
FIXED_CLUST_CONFIG = dict(clustering_algo="hdbscan",
                           cluster_params=json.dumps({"min_cluster_size": 15, "min_samples": 5}))

## Standard models (no instruction)

`all-MiniLM-L6-v2`, `all-mpnet-base-v2`, `BAAI/bge-m3`

In [ ]:
from sentence_transformers import SentenceTransformer

STANDARD_MODELS = [
    ("sentence-transformers/all-MiniLM-L6-v2", 384),
    ("sentence-transformers/all-mpnet-base-v2", 768),
    ("BAAI/bge-m3", 1024),
]

standard_results = []

for model_name, dim in STANDARD_MODELS:
    emb_path = embedding_path(cfg.cache_dir, model_name, cfg.sample_size, instruction="no_inst")

    if array_exists(emb_path):
        embeddings = load_array(emb_path)
        embed_time = 0.0
    else:
        t0 = time.time()
        st = SentenceTransformer(model_name, device=cfg.device)
        embeddings = st.encode(texts, batch_size=cfg.batch_size,
                               show_progress_bar=True, convert_to_numpy=True)
        embed_time = time.time() - t0
        save_array(emb_path, embeddings)

    reduced, labels, eval_time, coherence, entropy = evaluate_embeddings(
        embeddings, texts, stars, seed=cfg.seed, compute_coh=COMPUTE_COHERENCE)
    total_time = embed_time + eval_time
    metrics = compute_metrics(reduced, labels, runtime_s=total_time, seed=cfg.seed)
    standard_results.append({"model": model_name, "instruction": "no_inst",
                              **metrics, "coherence_cv": coherence, "rating_entropy": entropy})

    log_result(cfg.results_csv, {
        "pipeline": "custom",
        "sample_size": cfg.sample_size,
        "device": cfg.device,
        "embedding_model": model_name,
        "embedding_instruction": "no_inst",
        "embed_dim": dim,
        "embed_time_s": round(embed_time, 2),
        **FIXED_DR_CONFIG,
        **FIXED_CLUST_CONFIG,
        **metrics,
        "coherence_cv": coherence,
        "rating_entropy": entropy,
        "notes": "embedding comparison — fixed DR+clustering",
    })
    print(f"  {model_name:<55s}  sil={metrics['silhouette']}  "
          f"coh={coherence}  entr={entropy}  clusters={metrics['n_clusters']}")

## INSTRUCTOR models (instruction-tuned)

Tests `instructor-base` and `instructor-large` with all instruction variants.

In [ ]:
try:
    from InstructorEmbedding import INSTRUCTOR
    _has_instructor = True
except ImportError:
    print("Install: pip install InstructorEmbedding")
    _has_instructor = False

INSTRUCTOR_MODELS = [
    ("hkunlp/instructor-base", 768),
    ("hkunlp/instructor-large", 768),
    # ("hkunlp/instructor-xl", 768),  # GPU only
]
INSTRUCTOR_INSTRUCTIONS = ["no_inst", "generic", "domain", "sentiment"]

instructor_results = []

if _has_instructor:
    for model_name, dim in INSTRUCTOR_MODELS:
        inst_model = INSTRUCTOR(model_name)

        for instr_slug in INSTRUCTOR_INSTRUCTIONS:
            instr_text = INSTRUCTIONS[instr_slug]
            emb_path = embedding_path(cfg.cache_dir, model_name, cfg.sample_size,
                                       instruction=instr_slug)

            if array_exists(emb_path):
                embeddings = load_array(emb_path)
                embed_time = 0.0
            else:
                t0 = time.time()
                if instr_text:
                    pairs = [[instr_text, t] for t in texts]
                    embeddings = np.array(inst_model.encode(
                        pairs, batch_size=cfg.batch_size, show_progress_bar=True))
                else:
                    embeddings = np.array(inst_model.encode(
                        texts, batch_size=cfg.batch_size, show_progress_bar=True))
                embed_time = time.time() - t0
                save_array(emb_path, embeddings)

            reduced, labels, eval_time, coherence, entropy = evaluate_embeddings(
                embeddings, texts, stars, seed=cfg.seed, compute_coh=COMPUTE_COHERENCE)
            metrics = compute_metrics(reduced, labels,
                                      runtime_s=embed_time + eval_time, seed=cfg.seed)
            instructor_results.append({"model": model_name, "instruction": instr_slug,
                                        **metrics, "coherence_cv": coherence,
                                        "rating_entropy": entropy})

            log_result(cfg.results_csv, {
                "pipeline": "custom",
                "sample_size": cfg.sample_size,
                "device": cfg.device,
                "embedding_model": model_name,
                "embedding_instruction": instr_slug,
                "embed_dim": dim,
                "embed_time_s": round(embed_time, 2),
                **FIXED_DR_CONFIG,
                **FIXED_CLUST_CONFIG,
                **metrics,
                "coherence_cv": coherence,
                "rating_entropy": entropy,
                "notes": f"INSTRUCTOR comparison instr={instr_slug}",
            })
            print(f"  {model_name} | {instr_slug:<12s}  sil={metrics['silhouette']}  "
                  f"coh={coherence}  entr={entropy}  clusters={metrics['n_clusters']}")

## E5-instruct and Qwen3-Embedding (sentence-transformers prompt API)

In [ ]:
PROMPT_MODELS = [
    ("intfloat/multilingual-e5-large-instruct", 1024),
    ("Qwen/Qwen3-Embedding-0.6B", 1024),
    # GPU-only:
    # ("Qwen/Qwen3-Embedding-4B", 2560),
    # ("intfloat/e5-mistral-7b-instruct", 4096),
]
PROMPT_INSTRUCTIONS = ["no_inst", "generic", "domain"]

prompt_results = []

for model_name, dim in PROMPT_MODELS:
    try:
        st = SentenceTransformer(model_name, device=cfg.device)
    except Exception as e:
        print(f"Could not load {model_name}: {e}")
        continue

    for instr_slug in PROMPT_INSTRUCTIONS:
        instr_text = INSTRUCTIONS[instr_slug]
        emb_path = embedding_path(cfg.cache_dir, model_name, cfg.sample_size,
                                   instruction=instr_slug)

        if array_exists(emb_path):
            embeddings = load_array(emb_path)
            embed_time = 0.0
        else:
            t0 = time.time()
            encode_kwargs = dict(batch_size=cfg.batch_size, show_progress_bar=True,
                                  convert_to_numpy=True)
            if instr_text:
                encode_kwargs["prompt"] = instr_text
            embeddings = st.encode(texts, **encode_kwargs)
            embed_time = time.time() - t0
            save_array(emb_path, embeddings)

        reduced, labels, eval_time, coherence, entropy = evaluate_embeddings(
            embeddings, texts, stars, seed=cfg.seed, compute_coh=COMPUTE_COHERENCE)
        metrics = compute_metrics(reduced, labels,
                                  runtime_s=embed_time + eval_time, seed=cfg.seed)
        prompt_results.append({"model": model_name, "instruction": instr_slug,
                                **metrics, "coherence_cv": coherence,
                                "rating_entropy": entropy})

        log_result(cfg.results_csv, {
            "pipeline": "custom",
            "sample_size": cfg.sample_size,
            "device": cfg.device,
            "embedding_model": model_name,
            "embedding_instruction": instr_slug,
            "embed_dim": dim,
            "embed_time_s": round(embed_time, 2),
            **FIXED_DR_CONFIG,
            **FIXED_CLUST_CONFIG,
            **metrics,
            "coherence_cv": coherence,
            "rating_entropy": entropy,
            "notes": f"prompt-model comparison instr={instr_slug}",
        })
        print(f"  {model_name} | {instr_slug:<12s}  sil={metrics['silhouette']}  "
              f"coh={coherence}  entr={entropy}  clusters={metrics['n_clusters']}")

## Summary table

In [ ]:
import pandas as pd
from utils import load_results

df = load_results(cfg.results_csv)
emb_df = df[df["pipeline"] == "custom"].copy()

summary = (
    emb_df
    .sort_values("silhouette", ascending=False)
    [["embedding_model", "embedding_instruction", "n_clusters",
      "noise_ratio", "silhouette", "coherence_cv", "rating_entropy",
      "embed_time_s", "runtime_s"]]
    .reset_index(drop=True)
)
summary

## Speed vs. quality plot

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

for _, row in summary.iterrows():
    if pd.isna(row["silhouette"]):
        continue
    label = f"{row['embedding_model'].split('/')[-1]}\n{row['embedding_instruction']}"
    color = "steelblue" if pd.isna(row["coherence_cv"]) else plt.cm.RdYlGn(row["coherence_cv"])

    axes[0].scatter(row["embed_time_s"], row["silhouette"], s=80, color=color)
    axes[0].annotate(label, (row["embed_time_s"], row["silhouette"]),
                fontsize=7, xytext=(4, 4), textcoords="offset points")

axes[0].set_xlabel("Embedding time (s)")
axes[0].set_ylabel("Silhouette")
axes[0].set_title("Speed vs silhouette (colour = coherence, green=high)")
axes[0].grid(alpha=0.3)

# Silhouette vs coherence — true winners top-right
if summary["coherence_cv"].notna().any():
    for _, row in summary.iterrows():
        if pd.isna(row["silhouette"]) or pd.isna(row["coherence_cv"]):
            continue
        label = f"{row['embedding_model'].split('/')[-1]}\n{row['embedding_instruction']}"
        axes[1].scatter(row["silhouette"], row["coherence_cv"], s=80)
        axes[1].annotate(label, (row["silhouette"], row["coherence_cv"]),
                    fontsize=7, xytext=(4, 4), textcoords="offset points")
    axes[1].set_xlabel("Silhouette")
    axes[1].set_ylabel("C_v coherence")
    axes[1].set_title("Silhouette vs coherence — top-right = real winners")
    axes[1].grid(alpha=0.3)
else:
    axes[1].set_title("Coherence not computed (COMPUTE_COHERENCE=False)")

plt.tight_layout()
plt.show()

## Decision

| Winner | Model | Instruction | Silhouette | C_v coherence | Rating entropy | Reason |
|---|---|---|---|---|---|---|
| **Best overall** | _fill_ | _fill_ | _fill_ | _fill_ | _fill_ | |
| **Best CPU-fast** | _fill_ | _fill_ | _fill_ | _fill_ | _fill_ | |
| **Best multilingual** | _fill_ | _fill_ | _fill_ | _fill_ | _fill_ | |

Set `BEST_MODEL` and `BEST_INSTRUCTION` in `05_dimensionality_reduction.ipynb`.

Note: if an instruction-tuned model has the highest silhouette but equal or lower coherence
than a simpler model, prefer the simpler model — the instruction mechanically reshapes geometry
without adding topical information.